
### Before starting the tutorial, please set up your environment as follows:

```
>> python3 -m venv gf_venv
>> source gf_venv/bin/activate
>> python3 -m pip install --upgrade pip
>> python3 -m pip install  ipykernel  ipython  matplotlib  numpy  scipy  ipympl  ipywidgets
```









# Green's Function Hands-on Tutorial 


##### In this tutorial we will explore the fundamental properties of single-particle Green's functions, take a tour of some self-energy corrections and the physical scarios where they are relavent, and finally make conections to experimental spectra. 

### Task 1: Fundamental Properties

##### To make our calcuations concrete, we will use the Fermi electron gas to explore the various properties and use cases of Green's functions. This means, the free electron excitation dispersion $\varepsilon_{\mathbf{k}}$ is given by $\frac{\hbar\mathbf{k}^{2}}{2m}-\mu$ with a chemical potnetial $\mu$. Here, we will take $\hbar=1,~m=1$ for simplicity. 

> Plot the excitation dispersion and the density of states at $T=0$. Assume a 1D momentum space.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import spherical_jn
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams.update({'font.size': 15}) 

def ek(k_):
    mu = 2
    ek = (k_**2)
    return ek - mu

def calc_dos(k_,w_):
    dos_ = np.zeros((w_.shape[0]),dtype=float)
    dw_  = w_[1] - w_[0]
    for ik in np.arange(k_.shape[0]):
        iw = int(np.floor( (ek(k_[ik]) - np.min(w_))/(dw_) ))
        if iw < 0:
            iw = 0
        if iw >= w_.shape[0]:
            iw = w_.shape[0] - 1
        dos_[iw] = dos_[iw] + 1/dw_
    return dos_

k = np.arange(-np.pi,np.pi,0.001)
w = np.arange(-5,(np.pi)**2,0.1)

dos = calc_dos(k,w)

fig, axs = plt.subplots(1, 2, figsize=(6*1.5, 3*1.5))

axs[0].plot(k,ek(k),'r-',linewidth=2,label=r'$\varepsilon_{\mathbf{k}}$')
axs[1].plot(dos,w,'k-',linewidth=2)

axs[0].set_ylim([-5,np.max(w)])
axs[1].set_ylim([-5,np.max(w)])

axs[0].set_aspect('auto')
axs[1].set_aspect('auto')

axs[0].set_xticks([-np.pi,0,np.pi])
axs[0].set_xticklabels([r'$-\pi$',r'$0$',r'$\pi$'])
axs[1].set_yticks([])

axs[0].set_xlabel('k-space')
axs[0].set_ylabel('Energy [eV]')

axs[1].set_xlabel(r'DOS [eV$^{-1}$]')

axs[0].legend(frameon=False,loc=2,fontsize=15)

plt.tight_layout()
plt.show()

Questions: 
+ How was the DOS calculated? Which expansion of the delta function was used?
+ What is the trade off between the density of momentum and energy grids?
+ How does changing the density of points effect the smoothness of the DOS?
+ What is the origin of the peak in the dos at -2 eV? How is this demonstrated analytically?
+ How would you generate/plot the Fermi surface? 

> Calculate and plot the real & imaginary parts of $G^R_0(\mathbf{k},\omega)$ for the free Fermi gas. Additionally, plot a line-cut along energy for a given momentum. 

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import spherical_jn
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams.update({'font.size': 15}) 

def ek(k_):
    mu = 2
    ek = (k_**2)
    return ek - mu

def calc_GF0(k_,w_):
    eta_ = 0.1
    G0_  = np.zeros((k_.shape[0],w_.shape[0]),dtype=complex)
    for ik in np.arange(k_.shape[0]):
        G0_[ik,:]=1./(w_ - ek(k_[ik]) + 1j * eta_)
    return G0_

k = np.arange(-np.pi,np.pi,0.001)
w = np.arange(-5,(np.pi)**2,0.1)
Ik = int(k.shape[0]*2/3)

GF_0 = calc_GF0(k,w)

ImGF0 = -1.0*np.imag(GF_0.T)
ReGF0 = np.real(GF_0.T) 

fig, axs = plt.subplots(1, 3, figsize=(6*1.5*3/2, 3*1.5))

myextent=[np.min(k),np.max(k),np.min(w),np.max(w)]
im1 = axs[0].imshow(ImGF0,extent=myextent,cmap='bwr',origin='lower', vmin=-np.max(np.max(ImGF0)),vmax=np.max(np.max(ImGF0)))
im2 = axs[1].imshow(ReGF0,extent=myextent,cmap='bwr',origin='lower', vmin= np.min(np.min(ReGF0)),vmax=np.max(np.max(ReGF0)))

divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im1, cax=cax)

divider = make_axes_locatable(axs[1])
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im2, cax=cax)

axs[0].plot([k[Ik],k[Ik]],[np.min(w),np.max(w)],c='grey',linestyle='--')
axs[1].plot([k[Ik],k[Ik]],[np.min(w),np.max(w)],c='grey',linestyle='--')

axs[2].plot(w,-1.0*ImGF0[:,Ik],'r-',linewidth=2,label=r'Im$G$')
axs[2].plot(w,ReGF0[:,Ik],'k-',linewidth=2,label=r'Re$G$')
axs[2].set_xlabel('Energy [eV]')
axs[2].legend(fancybox=False,frameon=False)
axs[2].set_ylabel(r'$G(k_0,\omega)$ [eV$^{-1}$]')

axs[0].set_ylim([-5,np.max(w)])
axs[1].set_ylim([-5,np.max(w)])

axs[0].set_xticks([-np.pi,0,np.pi])
axs[0].set_xticklabels([r'$-\pi$',r'$0$',r'$\pi$'])
axs[1].set_xticks([-np.pi,0,np.pi])
axs[1].set_xticklabels([r'$-\pi$',r'$0$',r'$\pi$'])

axs[0].set_xlabel('k-space')
axs[1].set_xlabel('k-space')
axs[0].set_ylabel('Energy [eV]')
axs[1].set_ylabel('Energy [eV]')

axs[0].set_aspect('auto')
axs[1].set_aspect('auto')

plt.tight_layout()
plt.show()

Questions: 
+ How does $ImG_0$ compare to the band structure above?
+ How can we understand the relationship between the real and imaginary parts of $G$? Hint: ${\displaystyle \oint {\frac {G(\omega ')}{\omega '-\omega }}\,d\omega '=0}$ and ${\displaystyle \lim _{\varepsilon \to 0^{+}}{\frac {1}{x\pm i\varepsilon }}=\mp i\pi \delta (x)+{\mathcal {P}}{{\Big (}{\frac {1}{x}}{\Big )}}}$


> (Optional) Numerically, show $ImG_0$ and $ReG_0$ are related via a Hilbert Transform.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import scipy
import numpy as np
from scipy.special import spherical_jn
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams.update({'font.size': 15}) 

def ek(k_):
    mu = 2
    ek = (k_**2)
    return ek - mu

def calc_GF0(k_,w_):
    eta_ = 0.1
    G0_  = np.zeros((k_.shape[0],w_.shape[0]),dtype=complex)
    for ik in np.arange(k_.shape[0]):
        G0_[ik,:]=1./(w_ - ek(k_[ik]) + 1j * eta_)
    return G0_

k = np.arange(-np.pi,np.pi,0.001)
w = np.arange(-(np.pi)**2,(np.pi)**2,0.1)
Ik = int(k.shape[0]*2/3)

GF_0 = calc_GF0(k,w)

ImGF0 = np.imag(GF_0[Ik,:])
ReGF0 = np.real(GF_0[Ik,:])

GF0_Hilbert = scipy.signal.hilbert(ImGF0)*1j # by scipy convention imag(hilbert) is the hilbert transform of with an extra -1, and real(hilbert) is the input, so multiplying by 1j give the correct mathematical defintion. 

ImGF0_Hilbert = np.imag(GF0_Hilbert)
ReGF0_Hilbert = np.real(GF0_Hilbert)

fig, axs = plt.subplots(1, 2, figsize=(6*1.8, 3*1.8))

axs[0].plot(w,ImGF0,'r-',linewidth=2,label=r'Im$G$')
axs[0].plot(w,ReGF0,'k-',linewidth=2,label=r'Re$G$')
axs[0].set_xlabel('Energy [eV]')
axs[0].legend(fancybox=False,frameon=False)
axs[0].set_ylabel(r'$G(k_0,\omega)$ [eV$^{-1}$]')
axs[0].set_title('Explicit Calculation')

axs[1].plot(w,ImGF0_Hilbert,'r-',linewidth=2,label=r'Im$G$')
axs[1].plot(w,ReGF0_Hilbert,'k-',linewidth=2,label=r'Re$G$')
axs[1].set_xlabel('Energy [eV]')
axs[1].legend(fancybox=False,frameon=False)
axs[1].set_ylabel(r'$G(k_0,\omega)$ [eV$^{-1}$]')
axs[1].set_title('Calculated via Hilbert Transform')

axs[0].set_aspect('auto')
axs[1].set_aspect('auto')

plt.tight_layout()
plt.show()


##### Let us now examine the effect of the small imaginary part $i\delta$. As we have seen in the notes (Page 38), $\delta$ is related to the lifetime of the quasiparticle excitation. 

> Calculate and plot the real & imaginary parts of $G^R_0(\mathbf{k},\omega)$ for the free Fermi gas, along with a line-cut along energy for a given momentum for differing values of $\delta$. 
 

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import spherical_jn
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams.update({'font.size': 15}) 

def ek(k_):
    mu = 2
    ek = (k_**2)
    return ek - mu

def calc_GF0(k_,w_):
    eta_ = 0.1 # <<------------- Change values
    G0_  = np.zeros((k_.shape[0],w_.shape[0]),dtype=complex)
    for ik in np.arange(k_.shape[0]):
        G0_[ik,:]=1./(w_ - ek(k_[ik]) + 1j * eta_)
    return G0_

k = np.arange(-np.pi,np.pi,0.001)
w = np.arange(-5,(np.pi)**2,0.1)
Ik = int(k.shape[0]*2/3)

GF_0 = calc_GF0(k,w)

ImGF0 = -1.0*np.imag(GF_0.T)
ReGF0 = np.real(GF_0.T) 

fig, axs = plt.subplots(1, 3, figsize=(6*1.5*3/2, 3*1.5))

myextent=[np.min(k),np.max(k),np.min(w),np.max(w)]
im1 = axs[0].imshow(ImGF0,extent=myextent,cmap='bwr',origin='lower', vmin=-np.max(np.max(ImGF0)),vmax=np.max(np.max(ImGF0)))
im2 = axs[1].imshow(ReGF0,extent=myextent,cmap='bwr',origin='lower', vmin= np.min(np.min(ReGF0)),vmax=np.max(np.max(ReGF0)))

divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im1, cax=cax)

divider = make_axes_locatable(axs[1])
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im2, cax=cax)

axs[0].plot([k[Ik],k[Ik]],[np.min(w),np.max(w)],c='grey',linestyle='--')
axs[1].plot([k[Ik],k[Ik]],[np.min(w),np.max(w)],c='grey',linestyle='--')

axs[2].plot(w,-1.0*ImGF0[:,Ik],'r-',linewidth=2,label=r'Im$G$')
axs[2].plot(w,ReGF0[:,Ik],'k-',linewidth=2,label=r'Re$G$')
axs[2].set_xlabel('Energy [eV]')
axs[2].legend(fancybox=False,frameon=False)
axs[2].set_ylabel(r'$G(k_0,\omega)$ [eV$^{-1}$]')

axs[0].set_ylim([-5,np.max(w)])
axs[1].set_ylim([-5,np.max(w)])

axs[0].set_xticks([-np.pi,0,np.pi])
axs[0].set_xticklabels([r'$-\pi$',r'$0$',r'$\pi$'])
axs[1].set_xticks([-np.pi,0,np.pi])
axs[1].set_xticklabels([r'$-\pi$',r'$0$',r'$\pi$'])

axs[0].set_xlabel('k-space')
axs[1].set_xlabel('k-space')
axs[0].set_ylabel('Energy [eV]')
axs[1].set_ylabel('Energy [eV]')

axs[0].set_aspect('auto')
axs[1].set_aspect('auto')

plt.tight_layout()
plt.show()

> For a specific k-point, calculate $G_0(\omega)$ and $G_0(t)$ for a set of $\delta$ values.

In [ ]:

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import spherical_jn
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams.update({'font.size': 15}) 

inv_ev2fs=0.6582

def ek(k_):
    mu = 2
    ek = (k_**2)
    return ek - mu

def calc_GF0(k_,w_,eta_):
    #eta_ = 0.08
    G0_  = np.zeros((k_.shape[0],w_.shape[0]),dtype=complex)
    for ik in np.arange(k_.shape[0]):
        G0_[ik,:]=1./(w_ - ek(k_[ik]) + 1j * eta_)
    return G0_

k = np.arange(-np.pi,np.pi,0.001)
Ik = int(k.shape[0]/2.)
Nt = 1000
dt = 1./(4*np.fabs(ek(k[Ik])))
time_axis = np.arange(0, Nt) * dt * inv_ev2fs # convert to fs
shifted_freq = np.fft.ifftshift(np.fft.fftfreq(Nt, d=dt))


fig, axs = plt.subplots(2, 2, figsize=(6*1.5, 3*1.5*2))

for eta in [0.01,0.04,0.08,0.12]:
    GF_0 = calc_GF0(k,shifted_freq,eta)
    GFt_0 = np.fft.fft(GF_0[Ik,:]) # numpy convension uses exp(-iwt) for fft, and exp(+iwt) for inverse fft

    ImGFt0 = np.imag(GFt_0)
    ReGFt0 = np.real(GFt_0)

    axs[0,0].plot( shifted_freq , np.imag(GF_0[Ik,:]),label=r'$\delta=$'+str(eta),linewidth=2) 
    axs[0,1].plot( shifted_freq , np.real(GF_0[Ik,:]),linewidth=2) 

    axs[1,0].plot(time_axis,ImGFt0,linewidth=2)
    axs[1,1].plot(time_axis,ReGFt0,linewidth=2)   

axs[0,0].legend(fancybox=False,frameon=False)
axs[0,0].set_aspect('auto')
axs[0,0].set_xlabel(r'$\omega$ [eV]')
axs[0,0].set_ylabel(r'Im$G(k_0,\omega)$ [eV$^{-1}$]')
axs[0,1].set_aspect('auto')
axs[0,1].set_xlabel(r'$\omega$ [eV]')
axs[0,1].set_ylabel(r'Re$G(k_0,\omega)$ [eV$^{-1}$]')
axs[1,0].set_aspect('auto')
axs[1,0].set_xlim([0,10])
axs[1,0].set_xlabel(r'$t$ [fs]')
axs[1,0].set_ylabel(r'Im$G(k_0,t)$ [eV$^{-1}$]')
axs[1,1].set_aspect('auto')
axs[1,1].set_xlim([0,10])
axs[1,1].set_xlabel(r'$t$ [fs]')
axs[1,1].set_ylabel(r'Re$G(k_0,t)$ [eV$^{-1}$]')

plt.tight_layout()
plt.show()





Questions: 
+ What is the effect of $\delta$ on $ImG_0$ and $ReG_0$?
+ Write down an expression for $ImG_0$, compare to the definition of a Lorentzian Function. 
    + What does $\delta$ parameterize?
+ How does $\delta$ change the time dependece of $G_0$?
+ what is the relationship between $\delta$ and the excitation lifetime? 
+ Using these results, how can we interpret the data in Fig. 7 of https://doi.org/10.1103/PhysRevB.107.155139?  


### Task 2: Self-Energy Corrections

##### We have seen that modifications of the free Green's function due to scattering, echange, and correction effects may be accounted for by the so-called self-energy, denoted by $\Sigma(\mathbf{k},\omega)$. 

> In general, $\Sigma$ is complex $Re\Sigma+iIm\Sigma$, using Dyson's equation $G=G_0 + G_0 \Sigma G$ for an expression for $G$?
> What are the real and imaginary parts of $G$? How are they related to $Re\Sigma$ and $Im\Sigma$?
> How do we expect the spectrum of $G$ ($ReG$ and $ImG$) to be modified?

A simple self-energy that yields free-particle-like properties is given by $\Sigma_{qp}(\mathbf{k},\omega)=(1-\frac{1}{Z_{\mathbf{k}}})\omega-\frac{\gamma_{\mathbf{k}}}{Z_{\mathbf{k}}}i$.

> Using $\Sigma_{qp}$ calculate and plot the real & imaginary parts of $G^R$ assuming $G^R_0$ is that of a free Fermi gas, $Z_{\mathbf{k}}\equiv Z$, and $\gamma_{\mathbf{k}}\equiv\gamma$. 
> Additionally, overlay $ReG$ and $ImG$ with the excitation dispersion $\varepsilon_{\mathbf{k}}$ and plot a line-cut along energy for a given momentum of $G$ and $\Sigma$.
> Adjust $Z$ and $\gamma$ to mark their effect on the spectrum.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import spherical_jn
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams.update({'font.size': 15}) 

def ek(k_):
    mu = 2
    ek = (k_**2)
    return ek - mu

def calc_GF0(k_,w_):
    eta_ = 0.1
    G0_  = np.zeros((k_.shape[0],w_.shape[0]),dtype=complex)
    for ik in np.arange(k_.shape[0]):
        G0_[ik,:]=1./(w_ - ek(k_[ik]) + 1j * eta_)
    return G0_

k = np.arange(-np.pi,np.pi,0.001)
w = np.arange(-5,(np.pi)**2,0.1)
Ik = int(k.shape[0]*2/3)

def Sigma(k_,w_):
    Z=0.5
    gamma=0.1
    S0=np.zeros((k_.shape[0],w_.shape[0]),dtype=complex)
    for ik in np.arange(k_.shape[0]):
        S0[ik,:]=(1.-(1./Z))*w-1j*gamma/Z
    return S0

GF_0 = calc_GF0(k,w)

S=Sigma(k,w)

G=np.zeros((k.shape[0],w.shape[0]),dtype=complex)
for ik in np.arange(k.shape[0]):
    for iw in np.arange(w.shape[0]):
        G[ik,iw]=( GF_0[ik,iw]**(-1) - S[ik,iw] )**(-1)


ImGF = -1.0*np.imag(G.T)
ReGF = np.real(G.T)

fig, axs = plt.subplots(1, 3, figsize=(6*1.5*3/2, 3*1.5))

myextent=[np.min(k),np.max(k),np.min(w),np.max(w)]
im1 = axs[0].imshow(ImGF,extent=myextent,cmap='bwr',origin='lower', vmin=-np.max(np.max(ImGF)),vmax=np.max(np.max(ImGF)))
im2 = axs[1].imshow(ReGF,extent=myextent,cmap='bwr',origin='lower', vmin= np.min(np.min(ReGF)),vmax=np.max(np.max(ReGF)))

divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im1, cax=cax)

divider = make_axes_locatable(axs[1])
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im2, cax=cax)

axs[0].plot(k,ek(k),c='k',linestyle=':')
axs[1].plot(k,ek(k),c='k',linestyle=':')
axs[0].plot([k[Ik],k[Ik]],[np.min(w),np.max(w)],c='grey',linestyle='--')
axs[1].plot([k[Ik],k[Ik]],[np.min(w),np.max(w)],c='grey',linestyle='--')

axs[2].plot(w,np.imag(S[Ik,:]),'r--',linewidth=2,label=r'Im$\Sigma$')
axs[2].plot(w,np.real(S[Ik,:]),'k--',linewidth=2,label=r'Re$\Sigma$')

axs[2].plot(w,-1.0*ImGF[:,Ik],'r-',linewidth=2,label=r'Im$G$')
axs[2].plot(w,ReGF[:,Ik],'k-',linewidth=2,label=r'Re$G$')
axs[2].set_xlabel('Energy [eV]')
axs[2].legend(fancybox=False,frameon=False)
axs[2].set_ylabel(r'$G(k_0,\omega)$ [eV$^{-1}$]')
axs[2].set_ylim([-6,6])

axs[0].set_ylim([-5,np.max(w)])
axs[1].set_ylim([-5,np.max(w)])

axs[0].set_xticks([-np.pi,0,np.pi])
axs[0].set_xticklabels([r'$-\pi$',r'$0$',r'$\pi$'])
axs[1].set_xticks([-np.pi,0,np.pi])
axs[1].set_xticklabels([r'$-\pi$',r'$0$',r'$\pi$'])

axs[0].set_xlabel('k-space')
axs[1].set_xlabel('k-space')
axs[0].set_ylabel('Energy [eV]')
axs[1].set_ylabel('Energy [eV]')

axs[0].set_aspect('auto')
axs[1].set_aspect('auto')

plt.tight_layout()
plt.show()




Questions: 
+ What is the effect of $Z$ and $\gamma$?
+ What properties of the quasiparticle do $Z$ and $\gamma$ modify?

Extra Questions: 
+ What is $A(\omega)$?
+ What is the value of $\int d\omega ~A(\omega)$ for $G$ and $G_0$?
    + Do the values differ? why or why not? 


##### More generally, $\Sigma$ can be written as $\Sigma(\mathbf{k},z)=\int d\omega \frac{\sigma(\omega)}{z-\omega}$ where $\sigma$ is similar to the spectral function for $G$. Here, we will choose a simpler form to explore. Specifically, that for a single pole: $\Sigma_{simple}(\mathbf{k},\omega)=\frac{a}{\omega-\omega_0}$

> Using $\Sigma_{simple}$ calculate and plot the real & imaginary parts of $G^R$ assuming $G^R_0$ is that of a free Fermi gas. 
> Additionally, overlay $ReG$ and $ImG$ with the excitation dispersion $\varepsilon_{\mathbf{k}}$, and plot $\sigma_{simple}$.
> Adjust $a$ and $\omega_0$ to mark their effect on the spectrum.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import spherical_jn
from mpl_toolkits.axes_grid1 import make_axes_locatable
plt.rcParams.update({'font.size': 15}) 

def ek(k_):
    mu = 2
    ek = (k_**2)
    return ek - mu

def calc_GF0(k_,w_):
    eta_ = 0.1
    G0_  = np.zeros((k_.shape[0],w_.shape[0]),dtype=complex)
    for ik in np.arange(k_.shape[0]):
        G0_[ik,:]=1./(w_ - ek(k_[ik]) + 1j * eta_)
    return G0_


k = np.arange(-np.pi,np.pi,0.001)
w = np.arange(-5,(np.pi)**2,0.1)
Ik = int(k.shape[0]*2/3)

def Sigma(k_,w_):
    a=0.75
    eb=4
    eta=0.25
    S0=np.zeros((k_.shape[0],w_.shape[0]),dtype=complex)
    for ik in np.arange(k_.shape[0]):
        S0[ik,:]=a*(1./(w-eb+1j*eta))
    return S0

GF_0 = calc_GF0(k,w)

S=Sigma(k,w)

G=np.zeros((k.shape[0],w.shape[0]),dtype=complex)
for ik in np.arange(k.shape[0]):
    for iw in np.arange(w.shape[0]):
        G[ik,iw]=( GF_0[ik,iw]**(-1) - S[ik,iw] )**(-1)


ImGF = -1.0*np.imag(G.T)
ReGF = np.real(G.T)

fig, axs = plt.subplots(1, 3, figsize=(6*1.5*3/2, 3*1.5))

myextent=[np.min(k),np.max(k),np.min(w),np.max(w)]
im1 = axs[0].imshow(ImGF,extent=myextent,cmap='bwr',origin='lower', vmin=-np.max(np.max(ImGF)),vmax=np.max(np.max(ImGF)))
im2 = axs[1].imshow(ReGF,extent=myextent,cmap='bwr',origin='lower', vmin= np.min(np.min(ReGF)),vmax=np.max(np.max(ReGF)))

divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im1, cax=cax)

divider = make_axes_locatable(axs[1])
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im2, cax=cax)

axs[0].plot(k,ek(k),c='k',linestyle=':')
axs[1].plot(k,ek(k),c='k',linestyle=':')

axs[2].plot(np.imag(S[Ik,:]),w,'r-',linewidth=2,label=r'Im$\Sigma$')
axs[2].plot(np.real(S[Ik,:]),w,'k-',linewidth=2,label=r'Re$\Sigma$')
axs[2].plot([0,0],[np.min(w),np.max(w)],c='Grey',linestyle='-',linewidth=1,zorder=-1)

axs[2].set_ylabel('Energy [eV]')
axs[2].set_xlabel(r'$\Sigma(k_0,\omega)$ [eV$^{-1}$]')
axs[2].set_ylim([-5,np.max(w)])

axs[0].set_ylim([-5,np.max(w)])
axs[1].set_ylim([-5,np.max(w)])

axs[0].set_xticks([-np.pi,0,np.pi])
axs[0].set_xticklabels([r'$-\pi$',r'$0$',r'$\pi$'])
axs[1].set_xticks([-np.pi,0,np.pi])
axs[1].set_xticklabels([r'$-\pi$',r'$0$',r'$\pi$'])

axs[0].set_xlabel('k-space')
axs[1].set_xlabel('k-space')
axs[0].set_ylabel('Energy [eV]')
axs[1].set_ylabel('Energy [eV]')

axs[0].set_aspect('auto')
axs[1].set_aspect('auto')

plt.tight_layout()
plt.show()



Questions: 
+ In what energy range does the approximet self-energy $\Sigma_{qp}$ make sense?
+ Describe the Physical meaning of the effect of $\Sigma$ on the spectrum of $G$.

##### Finally, we will consider an electron-phonon coupled system. The lowest order self-energy correction is the Fan-Migdal self energy: $\Sigma_{FM}(\mathbf{k},\omega)=\int d\mathbf{q}~g^2 \left( \frac{       n_B(\omega^{ph}_\mathbf{q}) + n_F(\varepsilon_\mathbf{k+q})          }{ \omega + \omega^{ph}_\mathbf{q} -   \varepsilon_\mathbf{k+q}     +i\delta    } + \frac{       n_B(\omega^{ph}_\mathbf{q}) + 1- n_F(\varepsilon_\mathbf{k+q})          }{ \omega - \omega^{ph}_\mathbf{q} -   \varepsilon_\mathbf{k+q}        +i\delta }          \right)$

> Using $\Sigma_{FM}$ calculate and plot the real & imaginary parts of $G^R$ assuming $G^R_0$ is that of a free Fermi gas. 
> Additionally, overlay $ReG$ and $ImG$ with the excitation dispersion $\varepsilon_{\mathbf{k}}$, and plot $\sigma_{FM}$.
> Adjust $g$, $T$, $w_{ph}$, $\delta$ to mark their effect on the spectrum.


In [ ]:

def fermi(x_,T_):
    kbt=T_*(1./11602.)
    return 0.5*(1-np.tanh(x_/(2*kbt)))

def bose(x_,T_):
    kbt=T_*(1./11602.)
    nb = 1./( np.exp(x_/kbt)-1 )
    if( np.fabs(x_/kbt) <= 1E-6 ):
        nb = 1./(x_/kbt)
    if( np.fabs(x_/kbt) < 1E-16 ) or ( x_/kbt > 20. ):
        nb=0
    return nb

def wph(q_):
    w_ph=0.75
    return w_ph

def ek(k_):
    mu=2
    return (k_**2)-mu

def calc_GF0(k_,w_):
    eta_ = 0.05
    G0_  = np.zeros((k_.shape[0],w_.shape[0]),dtype=complex)
    for ik in np.arange(k_.shape[0]):
        G0_[ik,:]=1./(w_ - ek(k_[ik]) + 1j * eta_)
    return G0_

def Sigma(k_,w_):
    g=1#1.6 #0.5
    T=0.01
    eta=0.03
    S0=np.zeros((k_.shape[0],w_.shape[0]),dtype=complex)
    for ik in np.arange(k_.shape[0]):
        for iq in np.arange(k_.shape[0]):
            S0[ik,:] = S0[ik,:] + (g**2)*( ( bose(wph(k_[iq]),T) + fermi(ek(k_[ik]+k_[iq]),T) )/(w_+wph(k_[iq])-ek(k_[ik]+k_[iq])+1j*eta) + ( bose(wph(k_[iq]),T) + 1 - fermi(ek(k_[ik]+k_[iq]),T) )/(w_-wph(k_[iq])-ek(k_[ik]+k_[iq])+1j*eta) )     
    return S0/k.shape[0]

k = np.arange(-np.pi,np.pi,0.01)
w = np.arange(-5,(np.pi)**2,0.01)
Ik = int(k.shape[0]*2/3)

GF_0 = calc_GF0(k,w)
S=Sigma(k,w)

G=np.zeros((k.shape[0],w.shape[0]),dtype=complex)
for ik in np.arange(k.shape[0]):
    for iw in np.arange(w.shape[0]):
        G[ik,iw]=( GF_0[ik,iw]**(-1) - S[ik,iw] +1j*(.02) )**(-1)

ImGF = -1.0*np.imag(G.T)
ReGF =      np.real(G.T)

ImS = np.imag(S[ik,iw])
ReS = np.real(S[ik,iw])



In [ ]:

fig, axs = plt.subplots(1, 3, figsize=(6*1.5*3/2, 3*1.5))

myextent=[np.min(k),np.max(k),np.min(w),np.max(w)]
im1 = axs[0].imshow(ImGF,extent=myextent,cmap='bwr',origin='lower', vmin=-np.max(np.max(ImGF)),vmax=np.max(np.max(ImGF)))
im2 = axs[1].imshow(ReGF,extent=myextent,cmap='bwr',origin='lower', vmin= np.min(np.min(ReGF)),vmax=np.max(np.max(ReGF)))

divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im1, cax=cax)

divider = make_axes_locatable(axs[1])
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im2, cax=cax)

axs[0].plot(k,ek(k),c='k',linestyle=':')
axs[1].plot(k,ek(k),c='k',linestyle=':')

axs[2].plot(np.imag(S[Ik,:]),w,'r-',linewidth=2,label=r'Im$\Sigma$')
axs[2].plot(np.real(S[Ik,:]),w,'k-',linewidth=2,label=r'Re$\Sigma$')
axs[2].plot([0,0],[np.min(w),np.max(w)],c='Grey',linestyle='-',linewidth=1,zorder=-1)

axs[2].set_ylabel('Energy [eV]')
axs[2].set_xlabel(r'$\Sigma(k_0,\omega)$ [eV$^{-1}$]')
axs[2].set_ylim([-5,np.max(w)])

axs[0].set_ylim([-5,np.max(w)])
axs[1].set_ylim([-5,np.max(w)])

axs[0].set_xticks([-np.pi,0,np.pi])
axs[0].set_xticklabels([r'$-\pi$',r'$0$',r'$\pi$'])
axs[1].set_xticks([-np.pi,0,np.pi])
axs[1].set_xticklabels([r'$-\pi$',r'$0$',r'$\pi$'])

axs[0].set_xlabel('k-space')
axs[1].set_xlabel('k-space')
axs[0].set_ylabel('Energy [eV]')
axs[1].set_ylabel('Energy [eV]')

axs[0].set_aspect('auto')
axs[1].set_aspect('auto')

plt.tight_layout()
plt.show()

In [ ]:

fig, axs = plt.subplots(1, 3, figsize=(6*1.5*3/2, 3*1.5))

myextent=[np.min(k),np.max(k),np.min(w),np.max(w)]
im1 = axs[0].imshow(ImGF,extent=myextent,cmap='bwr',origin='lower', vmin=-np.max(np.max(ImGF)),vmax=np.max(np.max(ImGF)))
im2 = axs[1].imshow(ReGF,extent=myextent,cmap='bwr',origin='lower', vmin= np.min(np.min(ReGF)),vmax=np.max(np.max(ReGF)))

divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im1, cax=cax)

divider = make_axes_locatable(axs[1])
cax = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im2, cax=cax)

axs[0].plot(k,ek(k),c='k',linestyle=':')
axs[1].plot(k,ek(k),c='k',linestyle=':')

axs[2].plot(np.imag(S[Ik,:]),w,'r-',linewidth=2,label=r'Im$\Sigma$')
axs[2].plot(np.real(S[Ik,:]),w,'k-',linewidth=2,label=r'Re$\Sigma$')
axs[2].plot([0,0],[np.min(w),np.max(w)],c='Grey',linestyle='-',linewidth=1,zorder=-1)

axs[2].set_ylabel('Energy [eV]')
axs[2].set_xlabel(r'$\Sigma(k_0,\omega)$ [eV$^{-1}$]')
axs[2].set_ylim([-1,0])

axs[0].set_xlim([0,2.5])
axs[0].set_ylim([-1,0])
axs[1].set_xlim([0,2.5])
axs[1].set_ylim([-1,0])

axs[0].set_xlabel('k-space')
axs[1].set_xlabel('k-space')
axs[0].set_ylabel('Energy [eV]')
axs[1].set_ylabel('Energy [eV]')

axs[0].set_aspect('auto')
axs[1].set_aspect('auto')

plt.tight_layout()
plt.show()

Questions: 
+ What processes is $\Sigma_{FM}$ describing?
    + What do $n_B + n_F$ and $n_B + 1- n_F$ do?
    + What states does the electron transfer energy to and from?
+ Is $\Sigma$ Hermitian?
    + if not, why? Hint: where else do non-Hermitian operators appear?
+ What is the effect of $\Sigma$ on the spectrum of $G$?
+ Do we such effects as these in experiments?